In [1]:
# 1 — сжатие(2) 
import os, json, zlib, bz2, lzma, base64
import pandas as pd
try:
    import brotli
except ImportError:
    brotli = None

# === опции ===
CONTENT_TYPE = "notebook"   # "notebook" или "dataframe"
NB_PATH      = "bootstrap_ab_test.ipynb"   # <-- укажите путь к ноутбуку
SOURCE_DF    = None          # если CONTENT_TYPE == "dataframe" — присвойте нужный df
# =============

_ALGO_NAMES = {"Z": "zlib", "B": "bz2", "L": "lzma", "R": "brotli"}

def _compress_best(raw):
    candidates = {"Z": zlib.compress(raw, level=9),
                  "B": bz2.compress(raw, compresslevel=9),
                  "L": lzma.compress(raw, preset=9 | lzma.PRESET_EXTREME)}
    if brotli is not None:
        candidates["R"] = brotli.compress(raw, quality=11)
    tag = min(candidates, key=lambda k: len(candidates[k]))
    return tag, candidates[tag]

if CONTENT_TYPE == "notebook":
    if not NB_PATH:
        raise ValueError("Укажите путь к ноутбуку в переменной NB_PATH")
    if not os.path.isfile(NB_PATH):
        raise FileNotFoundError(f"Файл не найден: {NB_PATH!r}")
    print(f"Ноутбук: {NB_PATH}")
    with open(NB_PATH, "r", encoding="utf-8") as f:
        nb = json.load(f)
    parts = []
    for i, cell in enumerate(nb["cells"], start=1):
        source = cell.get("source", "")
        if isinstance(source, list):
            source = "".join(source)
        parts.append(f"### CELL {i} [{cell['cell_type']}] ###\n{source}\n")
    raw = "".join(parts).encode("utf-8")
    content_tag = "N"
    print(f"Ячеек: {len(nb['cells'])}")

elif CONTENT_TYPE == "dataframe":
    if SOURCE_DF is None:
        raise ValueError("CONTENT_TYPE == 'dataframe', но SOURCE_DF не задан")
    raw = SOURCE_DF.to_csv(index=False).encode("utf-8")
    content_tag = "D"
    print(f"Датафрейм: {SOURCE_DF.shape[0]} строк, {SOURCE_DF.shape[1]} колонок")

else:
    raise ValueError(f"неизвестный CONTENT_TYPE: {CONTENT_TYPE!r}")

algo_tag, compressed = _compress_best(raw)
_qr_prefix  = content_tag + algo_tag
_qr_payload = base64.b64encode(compressed).decode("ascii")
print(f"Сжатие: {len(raw)} -> {len(compressed)} байт ({_ALGO_NAMES[algo_tag]}), base64: {len(_qr_payload)} симв.")

Ноутбук: bootstrap_ab_test.ipynb
Ячеек: 1
Сжатие: 9959 -> 3164 байт (bz2), base64: 4220 симв.


In [4]:
# 2 — партиции или QR-коды

import io
import qrcode
from IPython.display import display, Image

# === опция ===
OUTPUT_MODE = "text"   # "qr" или "text"
# =============

if OUTPUT_MODE == "qr":
    max_chunk = 2953 - len(f"{_qr_prefix}|001/001|")
elif OUTPUT_MODE == "text":
    max_chunk = 2950
else:
    raise ValueError(f"неизвестный OUTPUT_MODE: {OUTPUT_MODE!r}")

chunks = [_qr_payload[i:i + max_chunk] for i in range(0, len(_qr_payload), max_chunk)]
total = len(chunks)
print(f"Частей: {total}")

for idx, chunk in enumerate(chunks, start=1):
    part = f"{_qr_prefix}|{idx:03d}/{total:03d}|{chunk}"
    print(f"--- часть {idx}/{total} ---")
    if OUTPUT_MODE == "qr":
        qr = qrcode.QRCode(error_correction=qrcode.constants.ERROR_CORRECT_L, box_size=8, border=4)
        qr.add_data(part.encode("ascii"))
        qr.make(fit=True)
        img = qr.make_image(fill_color="black", back_color="white")
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        display(Image(data=buf.getvalue()))
    else:
        print(part)

Частей: 2
--- часть 1/2 ---
NB|001/002|QlpoOTFBWSZTWcAr2AAAA0f/+0T/0Hh7//cff+/ePr////9/77RA4gB//+AAEABgDTx8wg5juu7ipLs0AAAAAAABWgFAGmkEEmTE9AoepskyepiaG1PUGh5QBpp6mjRpiaDQDQA0BoAAACqn4TNKnpk/VMoANBoAAAAGgADQAAAAAZAAAAEGmJgEwBMEaaYABMAJoaGJgAAAAEYRgEaYmQwBpkk01MVH6jU0aeoAA0AAAAAepo9IDTQ0BoAHqNAAAAaEGmJgEwBMEaaYABMAJoaGJgAAAAEYRgEaYmQwBIiEBAATQTU2RU8p5PUYBBAAHqHqAaaZAGgAZ6pp5QBoPUNPUvWAZm8WDiYpXRuPnbYKKSRbD7CvBySBYNokEEYA1rB2wKFoyvytDCD6mqQdGKmJIHKrQETGMXH2w1MQSCTZURiJQlNXCuNFvVSaBOMUYskCwN+QnJQTaunaMbA/kYtCVvXjt/53H3eq6yCx8/9j9yUj3gTExpCEKqVmkgXwaoS7Wn9H6ciC5XMT/Zj/x0NLgUausXhFADYUItv46p37hqPrMvbFb749SiDUXTx8VNeF7NB9mOT97d2HYCOLOwC6kgunQppVByg1L+GGzQwe9y83n6WdVhQmPA7UoMZv7Tn41d1e3OkvMCZG6K2x+nZdOMSzDMO6lPZWjCOY+vr4BNnM+80bTx1pT0UYRUgZuLGGXKkLAxVQgmBd7YOe0AX6z+nXQs1e17jzaJgyHMuiYusmsn0mZPW1BWspSuHyRVjRxqW6ePuUX5B6ltao8IWJTOYddkylO2uX4Wfn0V9GkuTk81b3421fB7V9qm02PYF7S7ZXbOTEdT3F+P0JvF3NaHneS60vZUz8llJ4jVW07NRDdA59UVqeSUQ1Zh73tdf1r7WMIs5Lz39Emq6Q8j8/1WQHAt3PkqDlN9UU7aRaywIZUpCDJMrYCzVOIGuau